# Cryptography-01 — Hachage, chiffrement symétrique et HMAC

**Série Cryptography** · Carnet 01 — Hachage, chiffrement symétrique et HMAC

La cryptographie protège des informations en transit ou au repos. Ce carnet ne
traite pas les algorithmes comme des boîtes noires : il les **exécute** avec la
bibliothèque native de .NET (`System.Security.Cryptography`) et montre, sur des
valeurs réelles, *ce que chaque brique fait* — une empreinte qui « explose »
quand on change un bit, un chiffrement qui ne doit son secret qu'à la clé, un
hachage à clé qui distingue le vrai du faux émetteur.


## Enjeu — protéger l'information, en le démontrant

Trois problèmes distincts, trois outils :

- **l'intégrité / l'identité d'un message** → la **fonction de hachage** ;
- **la confidentialité** (que seul le destinataire lise) → le
  **chiffrement symétrique** ;
- **l'authenticité** (savoir qui a émis) → l'**HMAC** (hachage à clé).

On exécute chaque outil et on mesure son comportement caractéristique, au lieu
de le décrire.


## 0. La fonction de hachage (SHA-256)

Une fonction de hachage `H` transforme un message de taille quelconque en une
**empreinte** de taille fixe (256 bits ici). Deux propriétés la rendent utile :
elle est **déterministe** (même message → même empreinte) et elle a un
**effet avalanche** — changer *un* bit du message change *la moitié* des bits de
l'empreinte.


In [1]:
using System;
using System.Linq;
using System.Security.Cryptography;
using System.Text;

// --- Fonction de hachage : SHA-256 ---
byte[] Hacher(byte[] msg)
{
    using var sha = SHA256.Create();
    return sha.ComputeHash(msg);
}
string EnHex(byte[] b) => BitConverter.ToString(b).Replace("-", "");

// Determinisme : meme message -> meme empreinte.
var m1 = Encoding.UTF8.GetBytes("La confidentialite est une propriete mesurable");
var m2 = Encoding.UTF8.GetBytes("La confidentialite est une propriete mesurable");
var h1 = Hacher(m1);
var h2 = Hacher(m2);
Console.WriteLine($"H(m)  = {EnHex(h1)}");
Console.WriteLine($"Determinisme : {(h1.SequenceEqual(h2) ? "OUI (meme empreinte)" : "NON")}");

// Effet avalanche : on bascule 1 bit d'entree, combien de bits d'empreinte changent ?
var m3 = (byte[])m1.Clone();
m3[0] ^= 0x01;                        // bascule 1 bit
var h3 = Hacher(m3);
int bitsChanges = 0;
for (int i = 0; i < h1.Length; i++)
{
    byte d = (byte)(h1[i] ^ h3[i]);          // difference bit a bit
    while (d != 0) { bitsChanges += d & 1; d >>= 1; }   // nombre de bits poses
}
Console.WriteLine();
Console.WriteLine($"Effet avalanche (1 bit d'entree) : {bitsChanges} / 256 bits d'empreinte changent");
Console.WriteLine($"Fraction : {bitsChanges} / 256 sur ce tirage  (~50 % en moyenne sur de nombreux tirages)");


The below script needs to be able to find the current output cell; this is an easy method to get it.

H(m)  = 275FE04BE0E4D2F5BB6482AD0E4D61830C99AADE36C445BE6B9517A45A0CCF3A


Determinisme : OUI (meme empreinte)


Effet avalanche (1 bit d'entree) : 106 / 256 bits d'empreinte changent


Fraction : 106 / 256 sur ce tirage  (~50 % en moyenne sur de nombreux tirages)


### Lecture du résultat

L'empreinte est **stable** pour un message donné (déterminisme) mais **fragile**
devant la plus petite modification : un bit d'entrée retourné fait basculer
environ la moitié des bits de sortie. C'est l'**effet avalanche** — la signature
d'une bonne fonction de hachage. Il rend pratiquement impossible de *construire*
deux messages au même hash, et de *retrouver* le message depuis son hash.


## 1. Le chiffrement symétrique (AES) — le secret, c'est la clé

Le chiffrement **symétrique** utilise la **même clé** pour chiffrer et
déchiffrer. L'algorithme (AES) est **public** — tout le monde le connaît. La
sécurité ne vient donc pas de l'algorithme, mais de la **clé** qu'on garde
secrète (principe de Shannon). On le démontre : même texte chiffré avec deux
clés différentes donne deux chiffrs différents.


In [2]:
// --- Chiffrement symetrique : AES ---
byte[] ChiffrerAes(byte[] cle, byte[] texte)
{
    using var aes = Aes.Create();
    aes.Key = cle;
    aes.Mode = CipherMode.ECB;            // ECB : pas d'IV, simple a demontrer
    aes.Padding = PaddingMode.PKCS7;
    using var enc = aes.CreateEncryptor();
    return enc.TransformFinalBlock(texte, 0, texte.Length);
}
string DecrypterAes(byte[] cle, byte[] chiffr)
{
    using var aes = Aes.Create();
    aes.Key = cle;
    aes.Mode = CipherMode.ECB;
    aes.Padding = PaddingMode.PKCS7;
    using var dec = aes.CreateDecryptor();
    return Encoding.UTF8.GetString(dec.TransformFinalBlock(chiffr, 0, chiffr.Length));
}

var texte = Encoding.UTF8.GetBytes("Message sensible a proteger");
var cleA = new byte[16]; for (int i = 0; i < 16; i++) cleA[i] = (byte)i;        // cles 128 bits
var cleB = new byte[16]; for (int i = 0; i < 16; i++) cleB[i] = (byte)(255 - i);

var chiffrA = ChiffrerAes(cleA, texte);
var chiffrB = ChiffrerAes(cleB, texte);
Console.WriteLine($"Chiffr cle A : {EnHex(chiffrA)}");
Console.WriteLine($"Chiffr cle B : {EnHex(chiffrB)}");
Console.WriteLine($"Meme texte, 2 cles -> chiffrs differents : {!(chiffrA.SequenceEqual(chiffrB))}");

var retour = DecrypterAes(cleA, chiffrA);
Console.WriteLine($"Retour (dechiffrage) : {retour}  ->  identique : {retour == "Message sensible a proteger"}");


Chiffr cle A : 48669D1A7F82D57BC19C1D619F5BA0636ED267BC8430059ECD10F169E2BECE41


Chiffr cle B : D589DF7A2DFFD79331DF4345AE9F2B81602BDB2380DE80961A20ECD08E50E273


Meme texte, 2 cles -> chiffrs differents : True


Retour (dechiffrage) : Message sensible a proteger  ->  identique : True


### Lecture du résultat

Le même message, chiffré avec deux clés différentes, produit **deux chiffrs
différents** — et le déchiffrement avec la bonne clé restitue exactement le
texte d'origine. L'algorithme AES est public et identique des deux côtés : ce
qui change le résultat, c'est **uniquement la clé**. La sécurité repose donc
entièrement sur le secret de la clé, jamais sur la discrétion de l'algorithme.


## 2. L'HMAC — le hachage *à clé*

Le hachage seul vérifie l'intégrité mais pas l'**authenticité** (qui a émis).
L'**HMAC** (Hash-based Message Authentication Code) combine une fonction de
hachage et une **clé secrète partagée** : seul qui possède la clé peut produire
un **tag** valide. C'est ce qui distingue un émetteur légitime d'un imposteur.


In [3]:
// --- HMAC : hachage A CLE (integrite + authenticite) ---
byte[] HmacSha256(byte[] cle, byte[] msg)
{
    using var h = new HMACSHA256(cle);
    return h.ComputeHash(msg);
}

var secret = new byte[32]; for (int i = 0; i < 32; i++) secret[i] = (byte)i;   // cle partagee
var msg = Encoding.UTF8.GetBytes("Paiement de 100 EUR");

var tagA = HmacSha256(secret, msg);
var cleAdversaire = new byte[32]; for (int i = 0; i < 32; i++) cleAdversaire[i] = (byte)(i + 1);
var tagB = HmacSha256(cleAdversaire, msg);

Console.WriteLine($"HMAC(cle secrete, m)    = {EnHex(tagA)}");
Console.WriteLine($"HMAC(cle adversaire, m) = {EnHex(tagB)}");
Console.WriteLine($"Meme message, 2 cles -> 2 tags differents : {!(tagA.SequenceEqual(tagB))}");

// Verification : avec la BONNE cle le tag recalcule colle ; avec la mauvaise, non.
var tagRecalcul = HmacSha256(secret, msg);
Console.WriteLine($"Verification (bonne cle)  : {tagRecalcul.SequenceEqual(tagA)}");
Console.WriteLine($"Verification (mauvaise cle): {tagRecalcul.SequenceEqual(tagB)}");


HMAC(cle secrete, m)    = 8A871547B9E56F9A094163FC8477DBF949C37537D7E207A6F207A7165177CF52


HMAC(cle adversaire, m) = 2377A5E1758241392749E6711E460BB063094B101A8D1BB917F1ADB24A4C9ADB


Meme message, 2 cles -> 2 tags differents : True


Verification (bonne cle)  : True


Verification (mauvaise cle): False


### Lecture du résultat

Même message, deux clés : **deux tags différents**. Le destinataire, qui
possède la clé secrète, recalcule le tag et le compare : s'il colle, le message
vient de l'émetteur légitime **et** n'a pas été altéré. L'HMAC fournit
d'un seul coup l'intégrité *et* l'authenticité — ce que le hachage brut ne
fournit pas.


## 3. Pourquoi le hachage *seul* ne suffit pas

C'est le piège classique : un hash « valide » ne prouve rien sur l'origine,
parce que **quiconque peut écrire le message peut aussi recalculer son hash**
— il n'y a pas de secret. L'attaquant modifie le message *et* son hash, et le
« contrôle d'intégrité » passe quand même.


In [4]:
// --- Pourquoi le hachage SEUL ne garantit pas l'authenticite ---
// Un attaquant qui peut REECRIRE le message peut aussi RECALCULER son hash
// (le hash n'a pas de secret). L'integrite seule ne fait pas l'authenticite.

var msgOrig = Encoding.UTF8.GetBytes("Virement de 100 EUR");
var hOrig   = Hacher(msgOrig);

// L'attaquant modifie le message ET recalcule le hash (aucun secret a deviner) :
var msgFaux = Encoding.UTF8.GetBytes("Virement de 100000 EUR");
var hFaux   = Hacher(msgFaux);
Console.WriteLine("Hash seul :");
Console.WriteLine($"  message modifie : {Encoding.UTF8.GetString(msgFaux)}");
Console.WriteLine($"  hash recalcule  : {EnHex(hFaux)}");
Console.WriteLine($"  -> le hash 'valide' le faux message (l'attaquant l'a recalcule) : OUI");

// Avec HMAC, l'attaquant ne peut PAS recalculer le tag sans la cle secrete :
var tagFaux     = HmacSha256(cleAdversaire, msgFaux);
var tagAttendu  = HmacSha256(secret, msgFaux);
Console.WriteLine();
Console.WriteLine("Avec HMAC :");
Console.WriteLine($"  tag calcule avec la MAUVAISE cle : {EnHex(tagFaux)}");
Console.WriteLine($"  le destinataire (bonne cle) le detecte : {!(tagFaux.SequenceEqual(tagAttendu))}");


Hash seul :


  message modifie : Virement de 100000 EUR


  hash recalcule  : 5459CBDADEB5EEBCBB94051B46E84B3C4D4FD03AD04AE4F9198708F1B2473CB0


  -> le hash 'valide' le faux message (l'attaquant l'a recalcule) : OUI


Avec HMAC :


  tag calcule avec la MAUVAISE cle : EC5EE335DAC83C2A02DC127226A1F7235317028645A6C2AEE67A5B27DE762B21


  le destinataire (bonne cle) le detecte : True


### Lecture du résultat

Le « contrôle d'intégrité » par hachage brut est **trompé** : l'attaquant a
modifié le montant *et* recalculé le hash, et le hash « valide ». Avec l'HMAC,
l'attaquant ne possède pas la clé secrète, donc il ne peut pas produire un tag
valide pour son faux message — le destinataire le **détecte**. La clé, pas
l'algorithme, fait la différence.


## Synthèse — trois problèmes, trois outils

| Problème | Outil | Ce qu'on a montré |
|---|---|---|
| Intégrité / identité | Hachage (SHA-256) | déterministe + effet avalanche (~50 % des bits en moyenne) |
| Confidentialité | Chiffrement symétrique (AES) | même texte + 2 clés = 2 chiffrs ; le secret est la clé |
| Authenticité | HMAC (hachage à clé) | même message + 2 clés = 2 tags ; le faux émetteur est détecté |
| Le piège | Hachage **sans** clé | un attaquant qui écrit le message recalcule son hash |

Le fil rouge : **l'algorithme est public, la clé est secrète.** Toute la
sécurité cryptographique moderne repose sur ce principe.


## Exercices


In [5]:
// Exercice 1 — L'IV (vecteur d'initialisation) et le mode CBC
// Ici on a utilise le mode ECB (pas d'IV) pour rester simple. Le mode CBC
// ajoute un IV aleatoire : ecrire pourquoi deux chiffrages du MEME texte avec
// le MEME algorithme et la MEME cle peuvent alors donner deux chiffrs
// DIFFERENTS, et a quoi sert l'IV (il doit etre PUBLIC, pas secrete).
void exercice_iv_cbc()
{
    // TODO etudiant
}


In [6]:
// Exercice 2 — Le chiffrement asymetrique (cles publique / privee)
// Le chiffrement symetrique partage UNE cle. Le chiffrement ASYMETRIQUE
// (RSA, ECDH) en utilise DEUX : ecrire la difference, et pourquoi on peut
// envoyer une cle symetrique a un inconnu en la chiffrant a SA cle publique
// (chiffrement hybride, comme TLS).
void exercice_asymetrique()
{
    // TODO etudiant
}


In [7]:
// Exercice 3 — La cote de securite en bits
// Une cle de 128 bits (AES) a une cote de 128 bits : la casser par force
// brute c'est 2^128 essais. Ecrire de combien d'essais il s'agit, pourquoi
// c'est hors de portee, et ce querepresentent 128 / 192 / 256 bits de cote.
void exercice_cote_bits()
{
    // TODO etudiant
}


## À retenir

- **La fonction de hachage** (SHA-256) est déterministe et a un **effet
  avalanche** : un bit d'entrée changé retourne ~50 % des bits de l'empreinte
  en moyenne (106 / 256 = 41,4 % sur le tirage de ce carnet).
- **Le chiffrement symétrique** (AES) : l'algorithme est **public**, la
  sécurité vient de la **clé** — même texte + 2 clés = 2 chiffrs.
- **L'HMAC** ajoute une **clé** au hachage : il fournit l'authenticité (pas
  seulement l'intégrité) — même message + 2 clés = 2 tags.
- **Le hachage seul ne fait pas l'authenticité** : un attaquant qui écrit le
  message recalcule son hash. La clé est ce qui fait la différence.


## Bibliographie

- NIST, *Secure Hash Standard (SHS)*, FIPS PUB 180-4 (août 2015) — les
  fonctions de hachage SHA-256/384/512. Archivé :

  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Cryptography\2015 - NIST - Secure Hash Standard (SHS) (FIPS 180-4).pdf` (sha8 `0455B406`).

- NIST, *Advanced Encryption Standard (AES)*, FIPS 197 (novembre 2001) — le
  chiffrement symétrique. Archivé :

  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Cryptography\2001 - NIST - Advanced Encryption Standard (FIPS 197).pdf` (sha8 `251DFE0B`).

- H. Krawczyk, M. Bellare, R. Canetti, *HMAC: Keyed-Hashing for Message
  Authentication*, RFC 2104 (février 1997) — l'HMAC. Archivé :

  `G:\Mon Drive\MyIA\IA\Bibliographie IA\Cryptography\1997 - Krawczyk Bellare Canetti - HMAC - Keyed-Hashing for Message Authentication (RFC 2104).txt` (sha8 `64D5245A`).

- Carnet 02 (à venir) : le chiffrement asymétrique et le schéma TLS.